# Avance 3: Baseline y Evaluación del Especialista Digital DISF

Este notebook establece el **Marco de Referencia (Baseline)** para evaluar la viabilidad de la extracción de información regulatoria. Siguiendo las instrucciones académicas, traducimos los conceptos clásicos de Machine Learning predictivo al entorno de Procesamiento de Lenguaje Natural (NLP) e IA Generativa.

# **Procesamiento de Lenguaje Natural**
## Maestría en Inteligencia Artificial Aplicada
### Tecnológico de Monterrey

* **Nombres y matrículas**
    * Sarmiento Cervantes Jacqueline: A01795863
    * Mayoral Terán Alexandro: A01795899
* **Número de equipo: 8**

## 1. Mapeo de Conceptos Académicos

* **Algoritmo Baseline:** *Zero-Shot Prompting* usando todo el texto del documento (*Full Context*).
* **Feature Importance:** Relevancia del contexto recuperado (Pesos de BM25 vs ChromaDB vía *Reciprocal Rank Fusion*).
* **Overfitting / Underfitting:** Alucinaciones (Overfitting) y Omisiones (Underfitting) en la generación del JSON de salida.
* **Métricas Adecuadas:** Calidad (Recall/Precision de campos extraídos frente al Golden Dataset) y Operativas (Latencia, Eficiencia de Tokens consumidos).
* **Desempeño Mínimo:** Recall > 85% y reducción de > 50% en consumo de Input Tokens vs el Baseline.

In [ ]:
import os
import sys
import pandas as pd
import json
from pathlib import Path

# Añadir el directorio raíz para importar módulos src
sys.path.insert(0, os.path.abspath('../'))

from src.nlp_core.agente import extraer_full_context, extraer_rag_simple
from src.nlp_core.schemas import RequerimientoInformacion
from dotenv import load_dotenv

load_dotenv()
print("✅ Entorno y dependencias inicializadas")

## 2. Definición del Golden Dataset y Corpus de Prueba
Cargaremos un fragmento de texto normativo y definiremos el "Golden Dataset" (el diccionario manual ideal) contra el cual mediremos la precisión y exhaustividad (Recall) de nuestro Agente.

In [ ]:
texto_prueba = """
Artículo 1. Las instituciones de crédito deberán enviar mensualmente un reporte de sus créditos comerciales a la DISF.
Dicho reporte, que denominaremos "Formulario de Créditos Comerciales Mensual", deberá contener:
1. Identificador del Crédito: Debe ser Alfanumérico con máximo 15 caracteres.
2. Moneda del crédito: Es obligatorio enviar la clave de moneda. Las opciones válidas son 'MXN' para Pesos Mexicanos y 'USD' para Dólares Estadounidenses.
3. Tasa de Interés: Debe ser un valor Numérico sin límite. Ojo, esta tasa no puede ser negativa en ningún caso.
"""

# Ground Truth (Golden Dataset Manual)
golden_dataset = {
    "nombre_formulario": "Formulario de Créditos Comerciales Mensual",
    "campos_esperados": ["Identificador del Crédito", "Moneda del crédito", "Tasa de Interés"]
}
print("✅ Ground Truth cargado exitosamente")

## 3. Ejecución del Baseline: Full Context
Enviamos todo el documento normativo crudo al LLM. Esto consume el máximo nivel de tokens posible y establece nuestra línea base de eficacia y gasto.

In [ ]:
print("Ejecutando Baseline (Full Context)...")
resultado_fc, telemetria_fc = extraer_full_context(texto_prueba)

print("\n--- 📊 Extracción Baseline ---")
print(f"Formulario Detectado: {resultado_fc.nombre_formulario}")
print("Campos Extraídos:")
for c in resultado_fc.campos_formulario:
    print(f" - {c.nombre_campo}")

print("\n--- ⏱️ Telemetría Baseline ---")
print(json.dumps(telemetria_fc, indent=2))

## 4. Ejecución del Modelo Avanzado: RAG Híbrido
Aquí se conectará el `MotorBusqueda` (que integra BM25 y ChromaDB con RRF). Al recuperar solo los fragmentos clave, esperamos bajar la latencia y los tokens drásticamente.

In [ ]:
query = "¿Cuáles son los campos requeridos para el reporte de créditos comerciales mensuales?"
print("Ejecutando RAG Avanzado...")

# NOTA: En sesiones futuras integraremos aquí el poblado automático de ChromaDB
# y la ejecución de extraer_rag_simple(query)
print("⏳ [Pendiente]: Poblar vectorstore localmente y ejecutar búsqueda RAG.")

## 5. Análisis de Telemetría y Trade-offs
Evaluaremos el costo operativo (Latencia y Tokens) de cada enfoque leyendo nuestra bitácora `telemetria_llm.jsonl`.

In [ ]:
log_path = Path("../data/03_output/telemetria_llm.jsonl")
if log_path.exists():
    df_telemetria = pd.read_json(log_path, lines=True)
    display(df_telemetria.tail(10))
    
    # --- ESPACIO PARA GRÁFICAS FUTURAS ---
    # ej: df_telemetria.groupby('estrategia')['total_tokens'].mean().plot(kind='bar')
else:
    print("El archivo de telemetría aún no ha sido creado. Ejecuta la extracción primero.")